In [2]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from config.settings import settings
import json,logging

In [4]:
logger=logging.getLogger(__name__)

In [5]:
import re

In [ ]:
async def rerank_chunks(
    query:str,
    chunks:list[dict],
    top_k:int=5,
)->list[dict]:
    if len(chunks)<=top_k:
        return chunks
    llm=ChatGoogleGenerativeAI(
        model=settings.GEMINI_MODEL,
        google_api_key=settings.GOOGLE_API_KEY,
        temperature=0.0,
    )
    chunks_text="\n---\n".join(
        f"Chunk {i}: {c.get('text', '')[:400]}" for i, c in enumerate(chunks[:15])
    )
    prompt=ChatPromptTemplate.from_messages([
        ("system","Score each chunk from 0 to 1 based on its relevance to the query and return  a JSON array:[{\"i\": 0, \"s\": 0.85}, ...]"),
        ("human", f"Query: {query}\n\n{chunks_text}"),
    ])
    try:
        result=await(prompt|llm).ainvoke({"query":query})
        content=result.content
        content=content.strip("` \n").removeprefix("json").strip()
        for i in json.loads(content):
           id=i.get("i",0)
           if 0<=id<len(chunks):
              chunks[id]["rerank_score"]=i.get("s",0.5)
        chunks.sort(key=lambda c:c.get("rerank_score",0),reverse=True)
        return chunks[:top_k]
    except Exception as e:
        logger.error(f"reranking failed: {e}")
        return chunks[:top_k]